# SVM Classification & Query Router

This notebook rebuilds the classical ML baseline for the project, incorporating the fixes and advanced experiments we developed:
1. **Grid Search**: Tuning TF-IDF and SVM hyperparameters across linear and RBF kernels.
2. **Learning Curves**: Visualizing the bias-variance tradeoff.
3. **Query Router**: A new task classifying short questions to simulate RAG intent routing.

## 1. Setup & Data Loading

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve, cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder

sns.set_theme(style='whitegrid')

In [ ]:
# Load the dataset
CHUNKS_PATH = Path("../../data/chunks.json")
if not CHUNKS_PATH.exists():
    CHUNKS_PATH = Path("data/chunks.json") # Fallback if run from root

with open(CHUNKS_PATH, 'r', encoding='utf-8') as f:
    chunks = json.load(f)

df = pd.DataFrame(chunks)
X = df['text'].astype(str).tolist()
y_raw = df['source'].astype(str).tolist()

le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = le.classes_
short_names = [n[:25] + '...' if len(n) > 28 else n for n in class_names]

print(f'Total chunks loaded: {len(X)}')
print(f'Number of classes: {len(class_names)}')

## 2. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

## 3. Advanced Grid Search

We use `n_jobs=None` to avoid the Windows multiprocessing bug when spawning processes from the Jupyter kernel.
We test both linear and RBF kernels.

In [ ]:
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, sublinear_tf=True)),
    ('svm', SVC(random_state=42))
])

param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1.0, 10.0]
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [1.0, 10.0],
        'svm__gamma': ['scale']
    }
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Starting Grid Search... (this will take a while)")
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring='f1_macro', n_jobs=None, verbose=2
)
grid_search.fit(X_train, y_train)

print(f"\nBest parameters: {grid_search.best_params_}")
print(f"Best CV F1-macro: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_

## 4. Final Evaluation & Confusion Matrix

In [ ]:
y_pred = best_model.predict(X_test)
print("Test Classification Report:")
print(classification_report(y_test, y_pred, target_names=short_names, digits=3))

fig, ax = plt.subplots(figsize=(14, 12))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=short_names, yticklabels=short_names)
ax.set_xlabel('Predicted Document', fontsize=12)
ax.set_ylabel('True Document', fontsize=12)
ax.set_title('SVM Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('svm_confusion_matrix.png', dpi=150)
plt.show()

## 5. Learning Curve
We visualize the training vs validation scores to check for overfitting or underfitting.

In [ ]:
print("Generating Learning Curve...")
train_sizes, train_scores, test_scores = learning_curve(
    best_model, X_train, y_train, cv=cv, scoring='f1_macro',
    n_jobs=None, train_sizes=np.linspace(0.1, 1.0, 5)
)

train_mean = np.mean(train_scores, axis=1)
test_mean = np.mean(test_scores, axis=1)

plt.figure(figsize=(10, 6))
plt.plot(train_sizes, train_mean, 'o-', color='r', label='Training score')
plt.plot(train_sizes, test_mean, 'o-', color='g', label='Cross-validation score')
plt.title('Learning Curve (SVM)')
plt.xlabel('Training Set Size')
plt.ylabel('F1-Macro Score')
plt.legend(loc='lower right')
plt.grid(True)
plt.savefig('svm_learning_curve.png', dpi=150)
plt.show()

---

## PART II: Query Router (Intent Classification)

Instead of classifying long document chunks, we will now build a classifier that maps **short user queries** to their relevant source document. This acts as a routing mechanism for a Retrieval-Augmented Generation (RAG) system.

**Dataset**: We use the synthetic questions generated in `qa_pairs.json`.

In [ ]:
print("--- QUERY ROUTER TASK ---")
QA_PATH = Path("../../data/qa_pairs.json")
if not QA_PATH.exists():
    QA_PATH = Path("data/qa_pairs.json")

with open(QA_PATH, 'r', encoding='utf-8') as f:
    qa_data = json.load(f)

q_texts = [item['question'] for item in qa_data]
q_labels = [item['source'] for item in qa_data]
q_y = le.transform(q_labels)

print(f"Loaded {len(qa_data)} short questions for intent routing.")

router_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True, strip_accents='unicode')),
    ('svm', SVC(kernel='linear', C=1.0, random_state=42))
])

cv_router = cross_validate(
    router_pipe, q_texts, q_y, cv=cv, scoring=['accuracy', 'f1_macro'], return_train_score=True
)

print(f"Query Router Val Accuracy: {cv_router['test_accuracy'].mean():.4f}")
print(f"Query Router Val F1-macro: {cv_router['test_f1_macro'].mean():.4f}")

y_pred_router = cross_val_predict(router_pipe, q_texts, q_y, cv=cv)

fig, ax = plt.subplots(figsize=(14, 12))
cm_router = confusion_matrix(q_y, y_pred_router)
sns.heatmap(cm_router, annot=True, fmt='d', cmap='Oranges', ax=ax,
            xticklabels=short_names, yticklabels=short_names)
ax.set_title('Query Router Confusion Matrix', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('router_confusion_matrix.png', dpi=150)
plt.show()